# Transformers with PyTorch — Sentiment Classification

LSTMs solved the vanishing gradient problem but still process tokens
*sequentially* — each step depends on the previous hidden state. This
serial dependency limits parallelism and makes training on long sequences slow.

**Transformers** (Vaswani et al., 2017) discard recurrence entirely.
Every position attends directly to every other position in a single
parallel operation — **self-attention**.

**Why self-attention beats recurrence for long-range dependencies:**
- RNN: information from step 1 reaches step 100 through 99 sequential
  multiplications → gradient degradation.
- Transformer: step 1 and step 100 are connected in *one* attention step.
  Distance doesn't matter.

**Architecture (encoder-only, like BERT):**
```
Token indices  (B, L)
  → Embedding + Positional Encoding   (B, L, d_model)
  → N × TransformerEncoderLayer:
        Multi-Head Self-Attention  →  Add & LayerNorm
        Feed-Forward Network       →  Add & LayerNorm
  → Mean Pool                         (B, d_model)
  → Linear → Sigmoid                  (B, 1)
```

> **Shape notation used throughout this notebook:**
> `B` = batch size (reviews per batch), `L` = sequence length (words per review),
> `d_model` = embedding / hidden dimension.

We use the same `movie_reviews` dataset as Lesson 03, so you can directly
compare Transformer vs CNN accuracy.

> 📖 [Alammar — The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)

## Step 1: Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import nltk
import random
from collections import Counter

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## Key Concept: Devices and `.to(device)`

PyTorch can run computations on the **CPU** or a **GPU** (NVIDIA CUDA or
Apple Silicon MPS). Moving a tensor or model to a device means its data and
computations live there.

```python
device = torch.device('mps' if torch.backends.mps.is_available()
                       else 'cuda' if torch.cuda.is_available()
                       else 'cpu')

model = MyModel().to(device)   # moves all weight tensors to device
X = X.to(device)               # moves input data to the same device
```

**Rule:** model and data must be on the *same* device — mixing them raises
a `RuntimeError`. Call `.to(device)` on both.

> 📖 [PyTorch — CUDA semantics](https://pytorch.org/docs/stable/notes/cuda.html)


## Key Concept: `Dataset` and `DataLoader`

`DataLoader` handles batching, shuffling, and parallel data loading so your
training loop stays simple.

```python
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)   # pairs inputs with labels
loader  = DataLoader(dataset,
                     batch_size=64,
                     shuffle=True)           # reshuffles every epoch

for Xb, yb in loader:                       # Xb: (64, ...) one mini-batch
    ...
```

Key parameters:
- `batch_size` — number of samples per gradient update
- `shuffle=True` — randomise order each epoch (always use for training)
- `num_workers` — background processes for loading (set to 2-4 on most machines)

> 📖 [PyTorch — Writing a Custom Dataset](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)


## Step 2: Load Movie Reviews (same as Lesson 03)

In [ ]:
from nltk.corpus import movie_reviews

random.seed(42)
docs = [(movie_reviews.words(fid), cat)
        for cat in movie_reviews.categories()
        for fid in movie_reviews.fileids(cat)]
random.shuffle(docs)

MAX_VOCAB = 10000
MAX_LEN   = 200

all_words = [w.lower() for ws, _ in docs for w in ws]
freq      = Counter(all_words)
vocab     = {'<PAD>': 0, '<UNK>': 1}
for w, _ in freq.most_common(MAX_VOCAB - 2):
    vocab[w] = len(vocab)

def encode(words, vocab, max_len):
    tokens = [vocab.get(w.lower(), 1) for w in words][:max_len]
    return tokens + [0] * (max_len - len(tokens))

X_data = [encode(ws, vocab, MAX_LEN) for ws, _ in docs]
y_data = [1 if label == 'pos' else 0 for _, label in docs]

X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor(y_data, dtype=torch.float32)
print('X:', X_tensor.shape, '  y:', y_tensor.shape)
print('Vocab size:', len(vocab))

## Step 3: DataLoaders

In [ ]:
split = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:split], X_tensor[split:]
y_train, y_test = y_tensor[:split], y_tensor[split:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=32)
print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')

## Key Concept: Self-Attention (Recap from Lesson 06)

You implemented scaled dot-product attention and multi-head self-attention from
scratch in Lesson 06 — refer back there for the full derivation.

In brief:
- **Queries, Keys, Values** — the input is projected three ways; similarity between
  Q and K determines how much each position attends to every other position
- **Scaling** — divide by `√d_k` to prevent softmax saturation
- **Multi-head** — run `H` independent attention functions and concatenate;
  each head can specialise in a different type of relationship

This lesson builds on that foundation and adds the components that make a
**full Transformer encoder**: positional encoding, a feed-forward sublayer,
residual connections, and layer normalisation.

## Step 4: Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, H, L_q, d_k)
    K: (B, H, L_k, d_k)
    V: (B, H, L_k, d_k)
    Returns: output (B, H, L_q, d_k), weights (B, H, L_q, L_k)
    """
    d_k = Q.size(-1)

    # TODO: compute scores = Q @ K^T / sqrt(d_k)
    # K.transpose(-2, -1) swaps the last two dims: (B,H,L_k,d_k) → (B,H,d_k,L_k)
    # scores shape should be (B, H, L_q, L_k)
    scores = ...

    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    # TODO: apply softmax over the last dimension (over keys)
    weights = ...

    # TODO: compute output = weights @ V
    # weights: (B, H, L_q, L_k)   V: (B, H, L_k, d_k)
    output = ...

    return output, weights


# Quick shape test
B, H, L, d_k = 2, 4, 10, 32
q = torch.randn(B, H, L, d_k)
k = torch.randn(B, H, L, d_k)
v = torch.randn(B, H, L, d_k)
out, w = scaled_dot_product_attention(q, k, v)
print('output:', out.shape)   # expect (2, 4, 10, 32)
print('weights:', w.shape)    # expect (2, 4, 10, 10)
print('weights sum:', w[0, 0, 0].sum().item())  # expect ~1.0

### Concept Check: Scaled Dot-Product Attention

**Q1.** The score matrix has shape `(B, H, L, L)`. What does entry `[b, h, i, j]`
represent? What does it mean if this value is very large?

**Q2.** We scale by `1/√d_k`. If `d_k = 64` and you did NOT scale, what would
happen to the softmax as `d_k` grows? (Hint: the variance of a dot product of
two random unit vectors grows linearly with dimension.)

**Q3.** After softmax, each row of the weight matrix sums to 1.
What is the output when the weight is `[1, 0, 0, ..., 0]` (all attention on
the first token)? When it is uniform `[1/L, 1/L, ...]`?

In [ ]:
# Q1:
# Q2:
# Q3:


## Key Concept: Multi-Head Attention (Recap from Lesson 06)

Same `MultiHeadSelfAttention` class from Lesson 06, re-defined here so this
notebook runs standalone.  See Lesson 06 for the full explanation of
`split_heads`, `merge_heads`, and the `W_o` output projection.

## Step 5: Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Four projection matrices — no bias (common in transformers)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        """Reshape (B, L, d_model) → (B, num_heads, L, d_k)."""
        B, L, _ = x.shape
        # TODO: view x as (B, L, num_heads, d_k) then transpose dims 1 and 2
        return ...

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)
        # TODO: project each of Q, K, V through their W matrices, then split_heads
        Q = ...    # (B, H, L, d_k)
        K = ...    # (B, H, L, d_k)
        V = ...    # (B, H, L, d_k)

        out, weights = scaled_dot_product_attention(Q, K, V, mask)

        # TODO: recombine heads → (B, L, d_model)
        # Steps: transpose(1,2) → contiguous() → view(B, -1, num_heads * d_k)
        out = ...

        return self.W_o(out), weights


# Quick shape test
mha = MultiHeadAttention(d_model=128, num_heads=4)
x   = torch.randn(2, 10, 128)          # (B=2, L=10, d_model=128)
out, w = mha(x, x, x)                  # self-attention: Q=K=V=x
print('MHA output:', out.shape)         # expect (2, 10, 128)
print('Attn weights:', w.shape)         # expect (2, 4, 10, 10)

### Concept Check: Multi-Head Attention

**Q1.** With `d_model=128` and `num_heads=4`, each head operates in `d_k=32`
dimensions. If instead `num_heads=8`, what is `d_k`? What is the total number
of parameters in `W_q` either way?

**Q2.** `split_heads` reshapes `(B, L, d_model)` → `(B, H, L, d_k)` using
`.view` followed by `.transpose(1, 2)`. Why is `.contiguous()` needed *after*
transposing when you later call `.view`?

**Q3.** The output projection `W_o` mixes information across heads.
What would happen if you removed `W_o` and just returned the concatenated
head outputs directly?

In [ ]:
# Q1:
# Q2:
# Q3:


## Key Concept: Positional Encoding

Self-attention is **permutation-invariant**: swapping two tokens produces the
same attention weights (just reordered). The model cannot tell whether "cat
bites dog" differs from "dog bites cat" without positional information.

**Solution:** Add a positional signal to the embedding before the first layer.
The "Attention Is All You Need" paper uses fixed sinusoidal encodings:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

**Why sinusoids?**
- Each position gets a unique `d_model`-dimensional fingerprint.
- The model can learn relative positions: `PE(pos + k)` is a *linear function*
  of `PE(pos)` for any fixed offset `k` — so relative distance is linearly
  decodable.
- Unlike learned positional embeddings, sinusoids generalise to sequence
  lengths not seen during training.

**Shape:** `pe` is a buffer of shape `(1, max_len, d_model)`.
It is added (broadcast) to the embedding: `x + pe[:, :L]`.


## Step 6: Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe  = torch.zeros(max_len, d_model)   # (max_len, d_model)

        # TODO: create pos — column vector of positions: shape (max_len, 1)
        pos = ...

        # TODO: create div — the denominator terms for each even embedding index
        # Formula: 10000^(2i/d_model) = exp(2i * -log(10000) / d_model)
        # Shape: (d_model // 2,)
        div = ...

        # TODO: fill pe with sin for even dims, cos for odd dims
        pe[:, 0::2] = ...   # shape: (max_len, d_model//2)
        pe[:, 1::2] = ...   # shape: (max_len, d_model//2)

        # Register as a non-trainable buffer — moves to GPU with the model
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):   # x: (B, L, d_model)
        # TODO: add positional encoding to x (broadcast over B), apply dropout
        return ...


# Quick test
pe_module = PositionalEncoding(d_model=128, max_len=200)
x_test    = torch.zeros(2, 50, 128)       # batch of 2, 50 tokens
x_out     = pe_module(x_test)
print('PE output shape:', x_out.shape)    # expect (2, 50, 128)
print('Non-zero (from PE):', (x_out != 0).sum().item())  # should be > 0

### Concept Check: Positional Encoding

**Q1.** A purely sinusoidal PE is *fixed* (not learned). What is one advantage
of this over learned positional embeddings? What is one disadvantage?

**Q2.** The PE buffer has shape `(1, max_len, d_model)` and is added to
embeddings of shape `(B, L, d_model)`. What broadcasting rule makes this work?

**Q3.** If you shuffled the tokens in a sentence and added the *same* shuffled
positional encodings, would the transformer produce the same output?
What if you shuffled tokens but kept positional encodings in original order?

In [ ]:
# Q1:
# Q2:
# Q3:


## Key Concept: Transformer Encoder Layer

Each encoder layer contains two sub-layers, each wrapped in a
**residual connection** and followed by **Layer Normalisation**:

```
Sub-layer 1:  x = LayerNorm(x + Dropout(MultiHeadAttention(x, x, x)))
Sub-layer 2:  x = LayerNorm(x + Dropout(FeedForward(x)))
```

**Why residual connections?**
`x + f(x)` lets gradients flow directly back through the addition operation
(gradient of addition w.r.t. both inputs is 1). Even if `f(x)` produces
near-zero gradients, the residual path keeps learning alive. This is what
makes 12-96 layer transformers trainable.

**Why LayerNorm and not BatchNorm?**
BatchNorm normalises across the *batch* dimension — its statistics depend on
batch size and order, which makes it unstable for variable-length sequences
(different positions would have different batch statistics).
LayerNorm normalises across the *feature* dimension for each token
independently — stable regardless of batch size or sequence length.

**Feed-Forward Network:**
`d_model → d_ff → d_model` (typically `d_ff = 4 × d_model`).
This per-position MLP is where most of the model's "knowledge" is stored.
In practice it accounts for ~2/3 of a transformer's parameters.


## Step 7: Transformer Encoder Layer

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):   # x: (B, L, d_model)
        # ── Sub-layer 1: self-attention + residual + LayerNorm ──
        # Self-attention: Q = K = V = x
        # TODO: call self.attn(x, x, x, mask), unpack output and weights
        attn_out, _ = ...
        # TODO: apply residual + dropout + LayerNorm
        # Pattern: x = self.norm1(x + self.drop(attn_out))
        x = ...

        # ── Sub-layer 2: feed-forward + residual + LayerNorm ──
        # TODO: apply self.ff, then residual + dropout + LayerNorm
        x = ...

        return x   # (B, L, d_model)


# Shape test
enc_layer = TransformerEncoderLayer(d_model=128, num_heads=4, d_ff=256)
x_in  = torch.randn(2, 50, 128)
x_out = enc_layer(x_in)
print('Encoder layer output:', x_out.shape)  # expect (2, 50, 128)

### Concept Check: Residuals & Layer Normalisation

**Q1.** The residual connection is `x = norm(x + f(x))` rather than
`x = f(x)`. During backpropagation, what is the gradient of the addition
w.r.t. the identity branch (the raw `x`)? Why does this prevent vanishing?

**Q2.** LayerNorm normalises each token's `d_model`-dimensional vector to
have mean 0 and variance 1. BatchNorm normalises across the batch dimension.
Give one concrete reason why BatchNorm is problematic for variable-length
text sequences.

**Q3.** The feed-forward network (FFN) is applied **independently to each
position** — it has no awareness of adjacent tokens. If attention already
mixes information across positions, what is the FFN's role?

In [ ]:
# Q1:
# Q2:
# Q3:


## Step 8: Full Sentiment Transformer

In [ ]:
class SentimentTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2,
                 d_ff=256, max_len=200, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.layers    = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):   # x: (B, L) token indices
        # TODO: embed tokens and add positional encoding
        emb = ...            # (B, L, d_model)

        # TODO: pass through each encoder layer in self.layers
        for layer in self.layers:
            emb = ...

        # TODO: mean-pool over the sequence length dimension (dim=1)
        # This reduces (B, L, d_model) → (B, d_model)
        pooled = ...

        # TODO: apply fc layer and sigmoid activation to get (B, 1)
        return ...


vocab_size = len(vocab)
model = SentimentTransformer(vocab_size).to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## Step 9: Train

In [ ]:
criterion    = nn.BCELoss()
optimizer    = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
# weight_decay adds L2 regularisation to the optimizer: effectively penalises
# large weights by subtracting a small fraction of them each step (helps prevent overfit)
train_losses = []

for epoch in range(10):
    model.train()
    total = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)   # Xb: (32,200)  yb: (32,)

        # Forward pass through: embedding → positional encoding → N encoder layers
        # → mean pool → classifier head. Multi-head attention scores recorded in graph.
        out  = model(Xb).squeeze()               # (32,1) → (32,) to match yb

        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()   # gradients flow through attention weights, FFN, embeddings
        optimizer.step()

        total += loss.item()
    avg = total / len(train_loader)
    train_losses.append(avg)
    print(f'Epoch {epoch+1:2d} | Loss: {avg:.4f}')

## Step 10: Evaluate

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        # TODO: get predictions, round to 0/1, count correct
        ...
print(f'Transformer Test accuracy: {correct/total*100:.1f}%')

## Step 11: Compare with CNN and Self-Attention (Lessons 03 & 06)

In [ ]:
# The CNN from Lesson 03 typically achieves ~82% on the same dataset.
# How does your Transformer compare?
# Consider: parameter count, training time, accuracy, attention interpretability.
print("Transformer accuracy above.  CNN (Lesson 03): ~82%")
print("\nParameter comparison:")
print(f"  Transformer: {sum(p.numel() for p in model.parameters()):,}")

if HAS_MPL and train_losses:
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_losses)+1), train_losses, 'o-',
             color='#a29bfe', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('BCE Loss')
    plt.title('Transformer — Training Loss (movie_reviews)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 12: Visualise Attention Weights

In [ ]:
def get_attention_weights(model, tokens, device):
    """Return (H, L, L) attention weights from the first encoder layer."""
    model.eval()
    with torch.no_grad():
        x   = torch.tensor([tokens], dtype=torch.long, device=device)
        emb = model.pos_enc(model.embedding(x))             # (1, L, d_model)
        _, weights = model.layers[0].attn(emb, emb, emb)    # (1, H, L, L)
    return weights[0].cpu()                                  # (H, L, L)

test_text  = "the film was absolutely wonderful and truly moving"
raw_tokens = [vocab.get(w, 1) for w in test_text.split()]
padded     = raw_tokens + [0] * (MAX_LEN - len(raw_tokens))
words      = test_text.split()
L          = len(words)

attn = get_attention_weights(model, padded, device)  # (H, L, L)

if HAS_MPL:
    fig, axes = plt.subplots(1, min(4, attn.size(0)), figsize=(14, 3.5))
    if attn.size(0) == 1:
        axes = [axes]
    for h, ax in enumerate(axes):
        mat = attn[h, :L, :L].numpy()
        im  = ax.imshow(mat, cmap='Blues', vmin=0)
        ax.set_xticks(range(L)); ax.set_xticklabels(words, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(L)); ax.set_yticklabels(words, fontsize=7)
        ax.set_title(f'Head {h+1}', fontsize=9)
    plt.suptitle('Self-Attention Weights — Layer 1 (row=query, col=key)', fontsize=11)
    plt.tight_layout()
    plt.show()

## Step 13: Visualise the Transformer Architecture

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_xlim(0, 9); ax.set_ylim(0, 11); ax.axis('off')
ax.set_title('Transformer Encoder Architecture', fontsize=13, fontweight='bold')

blocks = [
    (4.5, 0.6,  '(B, L) Token Indices',                '#dfe6e9'),
    (4.5, 1.9,  'Embedding\n+ Positional Encoding\n(B, L, d_model)', '#74b9ff'),
    (4.5, 3.5,  'Multi-Head\nSelf-Attention\n(B, L, d_model)',       '#a29bfe'),
    (4.5, 5.0,  'Add & LayerNorm',                     '#ffeaa7'),
    (4.5, 6.3,  'Feed-Forward\nd_model → d_ff → d_model\n(B, L, d_model)', '#55efc4'),
    (4.5, 7.8,  'Add & LayerNorm',                     '#ffeaa7'),
    (4.5, 9.2,  'Mean Pool  →  Linear  →  Sigmoid\n(B, 1)',          '#fd79a8'),
]
for x, y, label, color in blocks:
    rect = mpatches.FancyBboxPatch((x-2.0, y-0.55), 4.0, 1.1,
                                    boxstyle='round,pad=0.05',
                                    facecolor=color, edgecolor='#2d3436', lw=1.5)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5, fontweight='bold')

for i in range(len(blocks)-1):
    _, y0, _, _ = blocks[i]
    _, y1, _, _ = blocks[i+1]
    ax.annotate('', xy=(4.5, y1-0.55), xytext=(4.5, y0+0.55),
                arrowprops=dict(arrowstyle='->', color='#636e72', lw=1.5))

# Residual arrows
for (_, y_top, _, _), (_, y_bot, _, _), label in [
    (blocks[5], blocks[2], 'residual'),
    (blocks[7] if len(blocks) > 7 else blocks[5], blocks[4], 'residual'),
]:
    pass

ax.annotate('', xy=(1.8, 5.0), xytext=(1.8, 2.45),
            arrowprops=dict(arrowstyle='->', color='#e17055', lw=1.5,
                           connectionstyle='arc3,rad=0.0'))
ax.text(1.3, 3.7, 'residual', ha='center', fontsize=7.5, color='#e17055', rotation=90)

ax.annotate('', xy=(7.2, 7.8), xytext=(7.2, 5.55),
            arrowprops=dict(arrowstyle='->', color='#e17055', lw=1.5,
                           connectionstyle='arc3,rad=0.0'))
ax.text(7.7, 6.7, 'residual', ha='center', fontsize=7.5, color='#e17055', rotation=90)

ax.text(4.5, 10.5, '× N encoder layers (repeat blocks 3-8)', ha='center',
        fontsize=9, color='#636e72', style='italic')
plt.tight_layout()
plt.show()

### Final Concept Check: Transformers vs Recurrent Models

**Q1.** Self-attention has `O(L²)` memory complexity (the `L×L` score matrix).
For a sequence of length 1000, how much larger is the score matrix than for
length 100? Why is this a practical concern for document-level tasks?

**Q2.** LSTM processes tokens left-to-right, so the representation of token
`t` is conditioned only on tokens `1..t`. Transformer self-attention can
attend to tokens before *and* after `t` simultaneously. What does this
property enable that an LSTM encoder cannot do as directly?

**Q3.** You trained an encoder-only transformer for classification. Name one
task that would require an *encoder-decoder* transformer and explain what the
decoder adds.

**Q4.** Rank the four architectures (NN, CNN, RNN/LSTM, Transformer) by their
degree of **positional inductive bias** (from strongest to weakest). Justify.

In [ ]:
# Q1:
# Q2:
# Q3:
# Q4:
